# Week 12 live coding: MTPE workflow

Mục tiêu: đo hậu biên tập dịch máy bằng evidence dễ hiểu: time log, edit distance, revision labels và source note.

Core tuần này: chạy notebook, đọc effort table và revision-type table, xuất time figure, rồi viết caption, source note và Results paragraph có limitation.

In [1]:
import hashlib
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import rapidfuzz
    from rapidfuzz.distance import Levenshtein
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas==2.3.3", "matplotlib==3.9.4", "rapidfuzz==3.13.0"])
    import pandas as pd
    import matplotlib.pyplot as plt
    import rapidfuzz
    from rapidfuzz.distance import Levenshtein

plt.rcParams["svg.hashsalt"] = "week12-mtpe-workflow"

WEEK_PATH = Path("weeks/week-12-mtpe-workflow")
WEEK_DIR = WEEK_PATH if WEEK_PATH.exists() else Path(".")
DATA_PATH = WEEK_DIR / "data" / "raw" / "week12_mtpe_segments.csv"
TABLE_DIR = WEEK_DIR / "outputs" / "tables"
FIG_DIR = WEEK_DIR / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = "f4a22aed428056220b9617dd4259dbe7efa27a6f9308b525a4bd0032b19e3363"
REMOTE_DATA = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-12-mtpe-workflow/data/raw/week12_mtpe_segments.csv"
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(REMOTE_DATA, DATA_PATH)

actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Data file:", DATA_PATH)
print("SHA-256:", actual_sha)
if actual_sha != EXPECTED_SHA:
    print("Note: SHA differs from the course snapshot. Continue only if you intentionally changed the data.")

Data file: weeks/week-12-mtpe-workflow/data/raw/week12_mtpe_segments.csv
SHA-256: f4a22aed428056220b9617dd4259dbe7efa27a6f9308b525a4bd0032b19e3363


## 1. Read source, MT output, reference, and post-edit

Một row là một source segment được dịch bởi một synthetic MT profile, rồi được hậu biên tập thành `vi_postedit`. Cột `vi_reference` là bản dịch neo để so sánh nội dung; edit distance tuần này so sánh `vi_mt_output` với `vi_postedit`. Cột `pe_time_seconds` là time log synthetic để luyện phân tích, không phải dữ liệu thật của dịch giả.

In [2]:
data = pd.read_csv(DATA_PATH)
print("Rows:", len(data))
print("Source segments:", data["segment_id"].nunique())
print("Systems:", ", ".join(sorted(data["mt_system"].unique())))
print(data.loc[data["segment_id"].eq("P001"), ["segment_id", "mt_system", "zh_source", "vi_reference", "vi_mt_output", "vi_postedit", "pe_time_seconds"]].to_string(index=False))

Rows: 30
Source segments: 10
Systems: MT_A, MT_B, MT_C
segment_id mt_system            zh_source                                                                   vi_reference                                                                                  vi_mt_output                                                                      vi_postedit  pe_time_seconds
      P001      MT_A 推进教育数字化有助于扩大优质资源覆盖面。 Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao.                   Thúc đẩy giáo dục số hóa có ích cho mở rộng mặt bao phủ tài nguyên ưu chất.   Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao.              118
      P001      MT_B 推进教育数字化有助于扩大优质资源覆盖面。 Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao. Thúc đẩy chuyển đổi số trong giáo dục giúp mở rộng phạm vi tiếp cận nguồn lực chất lượng cao. Thúc đẩy số hóa giáo dục giúp mở rộng phạm vi tiếp cận nguồn lực chất lượng cao.               44
      

## 2. Warm-up before edit distance

Trước khi nhìn công thức, hãy kiểm tra bằng mắt: cột nào là raw MT output, cột nào là post-edited text, và cột nào chỉ là reference anchor?

In [3]:
print("Tiny example:", Levenshtein.distance("abc", "abdc"), "visible edit")

one_row = data.query("segment_id == 'P001' and mt_system == 'MT_B'")[[
    "zh_source", "vi_reference", "vi_mt_output", "vi_postedit", "revision_type", "severity"
]]
print(one_row.to_string(index=False))
print()
print("Edit distance compares vi_mt_output with vi_postedit. The reference helps us interpret meaning, but it is not the second string in this Week 12 calculation.")

Tiny example: 1 visible edit
           zh_source                                                                   vi_reference                                                                                  vi_mt_output                                                                      vi_postedit revision_type severity
推进教育数字化有助于扩大优质资源覆盖面。 Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao. Thúc đẩy chuyển đổi số trong giáo dục giúp mở rộng phạm vi tiếp cận nguồn lực chất lượng cao. Thúc đẩy số hóa giáo dục giúp mở rộng phạm vi tiếp cận nguồn lực chất lượng cao.   terminology    minor

Edit distance compares vi_mt_output with vi_postedit. The reference helps us interpret meaning, but it is not the second string in this Week 12 calculation.


## 3. Compute edit distance

`Levenshtein.distance(a, b)` đếm số lần thêm, xóa, hoặc thay ký tự để biến MT output thành post-edited text. Trong Core, học viên chỉ cần chạy và đọc output; tự sửa list-comprehension là Stretch. Đây là proxy bề mặt; nó không đo toàn bộ cognitive effort.

In [4]:
data["char_edit_distance"] = [
    Levenshtein.distance(mt, pe)
    for mt, pe in zip(data["vi_mt_output"], data["vi_postedit"])
]
data["normalized_edit_distance"] = [
    round(distance / max(len(mt), len(pe)), 3)
    for distance, mt, pe in zip(data["char_edit_distance"], data["vi_mt_output"], data["vi_postedit"])
]
data["revision_needed"] = data["severity"].ne("none")
print(data[["segment_id", "mt_system", "pe_time_seconds", "char_edit_distance", "normalized_edit_distance", "revision_type", "severity"]].head(9).to_string(index=False))

segment_id mt_system  pe_time_seconds  char_edit_distance  normalized_edit_distance revision_type severity
      P001      MT_A              118                  41                     0.526   terminology    major
      P001      MT_B               44                  16                     0.172   terminology    minor
      P001      MT_C              132                  25                     0.321      omission    major
      P002      MT_A              102                  29                     0.358   terminology    major
      P002      MT_B               38                   9                     0.105         style    minor
      P002      MT_C              119                  42                     0.519      omission    major
      P003      MT_A               86                  21                     0.273   terminology    minor
      P003      MT_B               29                   0                     0.000      no_error     none
      P003      MT_C              111

## 4. Summarize MTPE time by system

Bảng này là output chính của tuần. Đọc `mean_pe_time_seconds` trước, sau đó dùng edit distance và revision type để giải thích vì sao.

In [5]:
effort_summary = (
    data.groupby("mt_system", as_index=False)
    .agg(
        segments=("segment_id", "count"),
        mean_pe_time_seconds=("pe_time_seconds", "mean"),
        median_pe_time_seconds=("pe_time_seconds", "median"),
        mean_char_edit_distance=("char_edit_distance", "mean"),
        mean_normalized_edit_distance=("normalized_edit_distance", "mean"),
        revision_needed_rows=("revision_needed", "sum"),
    )
)
for col in ["mean_pe_time_seconds", "median_pe_time_seconds", "mean_char_edit_distance", "mean_normalized_edit_distance"]:
    effort_summary[col] = effort_summary[col].round(2)
effort_summary = effort_summary.sort_values("mean_pe_time_seconds")
effort_summary.to_csv(TABLE_DIR / "week12_mtpe_effort_summary.csv", index=False)
print(effort_summary.to_string(index=False))

mt_system  segments  mean_pe_time_seconds  median_pe_time_seconds  mean_char_edit_distance  mean_normalized_edit_distance  revision_needed_rows
     MT_B        10                  36.4                    35.0                      9.4                           0.11                     5
     MT_A        10                  99.4                   101.5                     33.9                           0.41                    10
     MT_C        10                 111.4                   117.5                     32.9                           0.40                    10


## 5. Revision labels explain the numbers

Time và edit distance nói “nhiều hay ít”. Revision labels nói “vì sao cần chỉnh”: `omission` = thiếu ý, `terminology` = sửa thuật ngữ, `word_order` = sửa trật tự câu tiếng Việt, `style` = làm mượt register/phrasing.

In [6]:
revision_summary = (
    data.groupby(["mt_system", "revision_type"], as_index=False)
    .size()
    .rename(columns={"size": "segment_count"})
    .sort_values(["mt_system", "segment_count"], ascending=[True, False])
)
revision_summary.to_csv(TABLE_DIR / "week12_revision_type_summary.csv", index=False)
print(revision_summary.to_string(index=False))

sample = data[data["segment_id"].isin(["P001", "P004", "P006", "P010"])]
sample.to_csv(TABLE_DIR / "week12_segment_edit_sample.csv", index=False)
print()
print("Segment sample:")
print(sample[["segment_id", "mt_system", "pe_time_seconds", "normalized_edit_distance", "revision_type", "post_editor_note"]].to_string(index=False))

mt_system revision_type  segment_count
     MT_A   terminology              6
     MT_A         style              2
     MT_A    word_order              2
     MT_B      no_error              5
     MT_B         style              3
     MT_B   terminology              2
     MT_C      omission             10

Segment sample:
segment_id mt_system  pe_time_seconds  normalized_edit_distance revision_type                                     post_editor_note
      P001      MT_A              118                     0.526   terminology                           term and word-order repair
      P001      MT_B               44                     0.172   terminology digitalization term adjusted to match policy wording
      P001      MT_C              132                     0.321      omission                       missing policy action restored
      P004      MT_A               94                     0.272   terminology           curriculum and competence wording repaired
      P004      

## 6. Export figures, settings, and submission text

Figure 1 is Core. Figure 2 is generated for comparison, but only submit it for Stretch. The notebook also exports a small Markdown file with the model caption, source note, and Results paragraph.

In [7]:
plt.rcParams.update({"font.size": 11, "axes.titlesize": 14, "axes.labelsize": 11})

plot = effort_summary.sort_values("mean_pe_time_seconds")
fig, ax = plt.subplots(figsize=(7.8, 4.8))
ax.barh(plot["mt_system"], plot["mean_pe_time_seconds"], color="#1f7a4d")
ax.set_title("Week 12 MTPE: mean post-editing time by system")
ax.set_xlabel("Mean post-editing time (seconds; lower is less effort)")
ax.set_ylabel("Synthetic MT profile")
for i, value in enumerate(plot["mean_pe_time_seconds"]):
    ax.text(value + 1.2, i, f"{value:.1f}s", va="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week12_time_by_system.png", dpi=200)
fig.savefig(FIG_DIR / "week12_time_by_system.svg", metadata={"Date": "2026-06-04"})
plt.close(fig)

plot = effort_summary.sort_values("mean_normalized_edit_distance")
fig, ax = plt.subplots(figsize=(7.8, 4.8))
ax.barh(plot["mt_system"], plot["mean_normalized_edit_distance"], color="#2f63ea")
ax.set_title("Week 12 MTPE: normalized edit distance by system")
ax.set_xlabel("Mean normalized character edit distance (lower is less editing)")
ax.set_ylabel("Synthetic MT profile")
for i, value in enumerate(plot["mean_normalized_edit_distance"]):
    ax.text(value + 0.01, i, f"{value:.2f}", va="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week12_edit_distance_by_system.png", dpi=200)
fig.savefig(FIG_DIR / "week12_edit_distance_by_system.svg", metadata={"Date": "2026-06-04"})
plt.close(fig)

domain_summary = (
    data.groupby("domain", as_index=False)
    .agg(
        rows=("segment_id", "count"),
        source_segments=("segment_id", "nunique"),
        mean_pe_time_seconds=("pe_time_seconds", "mean"),
        mean_normalized_edit_distance=("normalized_edit_distance", "mean"),
        revision_needed_rows=("revision_needed", "sum"),
    )
)
domain_summary[["mean_pe_time_seconds", "mean_normalized_edit_distance"]] = domain_summary[["mean_pe_time_seconds", "mean_normalized_edit_distance"]].round(2)
domain_summary.to_csv(TABLE_DIR / "week12_domain_effort_summary.csv", index=False)

metric_settings = pd.DataFrame([
    {"item": "rapidfuzz_version", "value": rapidfuzz.__version__},
    {"item": "distance_function", "value": "rapidfuzz.distance.Levenshtein.distance"},
    {"item": "normalization", "value": "char_edit_distance / max(len(vi_mt_output), len(vi_postedit))"},
    {"item": "time_measure", "value": "synthetic post-editing time in seconds for classroom practice"},
    {"item": "dataset_type", "value": "synthetic classroom dataset; not real post-editor logs"},
    {"item": "dataset_sha256", "value": EXPECTED_SHA},
])
metric_settings.to_csv(TABLE_DIR / "week12_metric_settings.csv", index=False)

print("Figures exported:")
print(FIG_DIR / "week12_time_by_system.png")
print(FIG_DIR / "week12_edit_distance_by_system.png")
print()
print("Metric settings:")
print(metric_settings.to_string(index=False))

Figures exported:
weeks/week-12-mtpe-workflow/outputs/figures/week12_time_by_system.png
weeks/week-12-mtpe-workflow/outputs/figures/week12_edit_distance_by_system.png

Metric settings:
             item                                                            value
rapidfuzz_version                                                           3.13.0
distance_function                          rapidfuzz.distance.Levenshtein.distance
    normalization    char_edit_distance / max(len(vi_mt_output), len(vi_postedit))
     time_measure    synthetic post-editing time in seconds for classroom practice
     dataset_type           synthetic classroom dataset; not real post-editor logs
   dataset_sha256 f4a22aed428056220b9617dd4259dbe7efa27a6f9308b525a4bd0032b19e3363


## 7. Paper-facing writing

Đoạn Results phải tách ba ý: synthetic time thấp/cao, revision type giải thích pattern, và limitation của synthetic data.

In [8]:
lowest = effort_summary.iloc[0]
highest_time = effort_summary.sort_values("mean_pe_time_seconds", ascending=False).iloc[0]
highest_edit = effort_summary.sort_values("mean_normalized_edit_distance", ascending=False).iloc[0]

paragraph = (
    f"In the synthetic Week 12 MTPE dataset, {lowest['mt_system']} had the lowest observed mean post-editing time "
    f"({lowest['mean_pe_time_seconds']} seconds across {int(lowest['segments'])} segments) and the lowest mean normalized edit distance "
    f"({lowest['mean_normalized_edit_distance']}). In contrast, {highest_time['mt_system']} had the highest mean time "
    f"({highest_time['mean_pe_time_seconds']} seconds); its edit distance was higher than {lowest['mt_system']}'s but slightly lower than "
    f"{highest_edit['mt_system']}'s. The revision-type table explains the pattern: lower-time rows often required no edit or only style repair, "
    "whereas higher-time rows often involved omission, terminology, or word-order repair. This is useful for a paper draft because it links a number to a visible revision reason and keeps the unit of analysis transparent. A cautious report should still avoid ranking real systems from this toy sample. However, the dataset is synthetic, the time values are classroom practice logs, and edit distance captures visible text change rather than all cognitive decisions made by a post-editor."
)
print(paragraph)
print()
print("Word count:", len(paragraph.split()))

caption = (
    "Figure 1 compares mean post-editing time across three synthetic MT profiles for 10 Chinese-Vietnamese education-policy source segments "
    "(30 system-segment rows). Lower time indicates less observed post-editing time in this classroom dataset."
)
source_note = (
    f"Source note: the synthetic dataset is data/raw/week12_mtpe_segments.csv, SHA-256 {EXPECTED_SHA}, with raw GitHub download from the course repository. "
    f"Edit distance was computed with RapidFuzz {rapidfuzz.__version__} Levenshtein distance. ISO 18587:2017 is used only as conceptual background for full human post-editing requirements, not as a claim that this classroom workflow implements the standard. "
    "For empirical MTPE context, see Sun, Wang, and Jia (2025), PLOS ONE, DOI 10.1371/journal.pone.0328511. Access date: 2026-06-04."
)
submission_text = f"""# Week 12 Submission Text Model

## Caption

{caption}

## Source Note

{source_note}

## Results Paragraph

{paragraph}
"""
(TABLE_DIR / "week12_submission_text.md").write_text(submission_text, encoding="utf-8")
print("Submission text exported:", TABLE_DIR / "week12_submission_text.md")
print()
print("Caption:")
print(caption)
print()
print("Source note:")
print(source_note)

In the synthetic Week 12 MTPE dataset, MT_B had the lowest observed mean post-editing time (36.4 seconds across 10 segments) and the lowest mean normalized edit distance (0.11). In contrast, MT_C had the highest mean time (111.4 seconds); its edit distance was higher than MT_B's but slightly lower than MT_A's. The revision-type table explains the pattern: lower-time rows often required no edit or only style repair, whereas higher-time rows often involved omission, terminology, or word-order repair. This is useful for a paper draft because it links a number to a visible revision reason and keeps the unit of analysis transparent. A cautious report should still avoid ranking real systems from this toy sample. However, the dataset is synthetic, the time values are classroom practice logs, and edit distance captures visible text change rather than all cognitive decisions made by a post-editor.

Word count: 141
Submission text exported: weeks/week-12-mtpe-workflow/outputs/tables/week12_submi